# Image fundamentals: from pixels to a measurable baseline

**Scenario:** isolate a bright component on a controlled inspection surface. The default path is deterministic, CPU-only, credential-free, and uses synthetic data with no privacy risk.

**Success criterion:** reach IoU ≥ 0.95 on the baseline scene, expose a failure under illumination shift, and test a bounded mitigation.

## 1. Environment and contracts

The lab expects a two-dimensional `float64` array with finite values in `[0, 1]`. Keeping this contract explicit prevents a common silent failure: mixing `uint8` `[0, 255]` images with normalized thresholds.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np

lesson_dir = Path.cwd()
if not (lesson_dir / 'lab.py').exists():
    lesson_dir = Path('curriculum/beginner/01-image-fundamentals')
spec = importlib.util.spec_from_file_location('image_fundamentals_lab', lesson_dir / 'lab.py')
lab = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
np.__version__

## 2. Inspect the image before transforming it

Shape, dtype, minimum, and maximum are part of the input contract—not debugging trivia. The generated target is retained separately so evaluation does not mistake the prediction for truth.

In [ ]:
image, target = lab.make_scene()
summary = {'shape': image.shape, 'dtype': str(image.dtype), 'min': float(image.min()), 'max': float(image.max()), 'foreground_pixels': int(target.sum())}
summary

## 3. Establish a baseline

A threshold of `0.50` lies between the expected background and foreground intensities. We calculate IoU, precision, and recall so over- and under-segmentation remain distinguishable.

In [ ]:
baseline_prediction = lab.segment(image, threshold=0.50)
baseline = lab.evaluate(baseline_prediction, target)
assert baseline.intersection_over_union >= 0.95
baseline

The assertion makes the lesson's success criterion executable. On the seeded scene, the bright foreground is cleanly separated from the background. This result applies only to the controlled distribution we created.

## 4. Experiment: change one decision variable

Sweep thresholds while holding the scene fixed. Low values invite false positives; high values eventually remove true foreground pixels.

In [ ]:
results = lab.sweep_thresholds(image, target, [0.30, 0.45, 0.60, 0.75])
[(threshold, round(metrics.intersection_over_union, 3), round(metrics.precision, 3), round(metrics.recall, 3)) for threshold, metrics in results]

## 5. Failure injection: illumination shift

Now reduce every intensity by 35%. The shape has not changed, but the numeric distribution has. This is a realistic class of camera-pipeline failure: an apparently sensible fixed threshold becomes miscalibrated.

In [ ]:
dark_image = np.clip(image * 0.65, 0.0, 1.0)
failed = lab.evaluate(lab.segment(dark_image, 0.50), target)
assert failed.recall < baseline.recall
failed

## 6. Bounded mitigation and comparison

For this controlled exercise, lower the threshold and measure the result. In production, recalibration must be based on representative validation data and paired with input monitoring; silently adapting on arbitrary live images can hide drift or amplify errors.

In [ ]:
mitigated = lab.evaluate(lab.segment(dark_image, 0.32), target)
assert mitigated.intersection_over_union > failed.intersection_over_union
{'baseline': baseline, 'shifted': failed, 'mitigated': mitigated}

## 7. Input-contract failure

The reusable function rejects values outside `[0, 1]`. Test the guard rather than relying on every caller to remember normalization.

In [ ]:
try:
    lab.segment((image * 255).astype(np.uint8), 0.50)
except ValueError as error:
    print(f'Expected contract failure: {error}')
else:
    raise AssertionError('uint8 range mismatch should have been rejected')

## 8. Production upgrade

| Notebook choice | Production upgrade |
| --- | --- |
| Synthetic scene | Versioned, representative and licensed evaluation data |
| One global threshold | Calibrated decision rule with per-condition release gates |
| In-memory arrays | Validated decoding, channel, dtype, range and metadata contracts |
| Aggregate metrics | Slice metrics, monitored input distributions and reviewable errors |
| Immediate replacement | Shadow test, staged rollout, rollback and safe fallback |

### Exercises

1. Increase the noise and find where no fixed threshold reaches the baseline criterion.
2. Implement RGB-to-luminance conversion with an explicit channel-order contract.
3. Add false-positive and false-negative counts to the metrics object.
4. Propose evaluation slices for two different cameras and day/night operation.
5. Explain what evidence would justify a learned segmentation model.